# Final review: 65 kelompok konflik Dataset V2

Review audit tersimpan saja. Tidak mengimpor modul persiapan dataset, tidak membuat
split baru, tidak mengubah label, dan tidak menjalankan model/training/final test.
Unggah `review_dataset_v2.py` dari workspace ke `/content/` melalui panel Files.
Gunakan runtime CPU. Notebook ini tidak mengambil kode dari GitHub.


In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
import json
import uuid
import importlib.util

drive.mount('/content/drive')
OUTPUT_ROOT = Path('/content/drive/MyDrive/fer_cnn_outputs')
AUDIT_DIR = None  # Isi Path('.../run_id/clean_grouped_v2') jika ada beberapa hasil.
DATA_DIR = None  # Gunakan Path('.../Data') jika sumber ekstrak masih tersedia.
ZIP_PATH = Path('/content/drive/MyDrive/FacialExpression-Dataset/archive (1).zip')
EXPECTED_SPLIT_SHA256 = '7e9742da12ed2caa4652e23af7a78215f57fd1e44e37cec2e7c70cd664b66ec8'

if AUDIT_DIR is None:
    matches = []
    for summary_path in OUTPUT_ROOT.glob('clean_grouped_v2_*/*/clean_grouped_v2/summary.json'):
        if json.loads(summary_path.read_text()).get('split_sha256') == EXPECTED_SPLIT_SHA256:
            matches.append(summary_path.parent)
    if len(matches) != 1:
        raise ValueError(f'Set AUDIT_DIR explicitly. Matching audit folders: {matches}')
    AUDIT_DIR = matches[0]
else:
    AUDIT_DIR = Path(AUDIT_DIR)

module_path = Path('/content/review_dataset_v2.py')
if not module_path.is_file():
    raise FileNotFoundError('Upload review_dataset_v2.py to /content first.')
spec = importlib.util.spec_from_file_location('saved_v2_review', module_path)
review = importlib.util.module_from_spec(spec)
spec.loader.exec_module(review)
print('Read-only audit:', AUDIT_DIR)


## Buat review konflik saja

Jika runtime lama masih aktif, isi `DATA_DIR` dengan folder `Data` yang digunakan
saat audit. Jika tidak, kode berikut memverifikasi arsip lalu mengekstrak hanya
373 sumber karantina ke folder sementara. Deteksi duplikat dan split tidak diulang.


In [ ]:
if DATA_DIR is None:
    DATA_DIR = review.extract_conflict_sources(AUDIT_DIR, ZIP_PATH, EXPECTED_SPLIT_SHA256)
REVIEW_DIR = AUDIT_DIR.parent / (
    'final_conflict_review_' + datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
    + '_' + uuid.uuid4().hex[:8])
checks = review.review_saved_audit(
    AUDIT_DIR, DATA_DIR, REVIEW_DIR,
    expected_split_sha256=EXPECTED_SPLIT_SHA256)
print(json.dumps(checks, indent=2))
print('Review outputs:', REVIEW_DIR)
import pandas as pd
from IPython.display import display
display(pd.read_csv(REVIEW_DIR / 'conflict_class_impact.csv'))
display(pd.read_csv(REVIEW_DIR / 'conflict_group_review.csv'))
if checks['conflict_images_unavailable_or_changed']:
    print('STOP visual approval: inspect conflict_image_status.csv for missing/changed sources.')


## Prosedur pemeriksaan manusia

1. Buka `index.html` beserta folder `conflict_contact_sheets/`, atau buka PNG
   tiap grup di Drive. Satu halaman hanya berisi satu grup, maksimal 9 gambar,
   panel gambar hingga 320x320; judul memuat group ID lengkap dan label asli.
2. Mulai dari `priority=first` di `conflict_group_review.csv` (12 grup), terutama
   grup berantai `isolate_0c7d7a51e5cc95b589e7ba0efe1fa9746ebc2c8b42e5dabd9c09694233572e2c`.
   Baca pasangan dan metrik terkait dalam `conflict_edges_review.csv`.
3. Periksa seluruh anggota: salinan foto sama, transformasi kecil, subjek sama
   dengan ekspresi berbeda, foto berbeda, atau tidak dapat dipastikan.
   Kesamaan subjek tidak cukup untuk menyimpulkan duplikat.
4. Isi salinan `review_decisions_template.csv`: reviewer, status
   `pending/reviewed/needs_second_review`, visual_relationship, recommendation
   `retain_quarantine/review_future_protocol/uncertain`, dan catatan berbasis path.
   Jangan menebak label yang benar atau memakai prediksi model.
5. Periksa sisa grup hingga seluruh 65 grup ditinjau; kelompok exact tetap
   perlu diperiksa bila ada anggota tambahan dari edge tidak pasti.
6. Tinjau dampak per kelas, terutama Fear, Angry dan Surprise. Jika satu edge
   salah menyatukan foto/ekspresi berbeda, catat revisi untuk protokol versi baru.
   Jangan mengubah label, manifest keputusan atau split audit ini.
7. Pertahankan hash lama. Persetujuan visual belum otomatis berarti persetujuan
   training; notebook review tidak menetapkan REVIEWED_V2_SPLIT_SHA256.
